# Notebook 03 — Privacy, Governance & Compliance Audit
## NovaCred Credit Application Dataset
### Role: Data Governance Officer

---

This notebook is the formal governance and compliance audit for NovaCred's credit application pipeline. It is structured around five responsibilities of the Governance Officer role:

1. **PII Identification & Classification** — cataloguing every personally identifiable field and its identification risk level
2. **Pseudonymisation Demonstration** — implementing and critically evaluating privacy-preserving techniques in code
3. **Regulatory Compliance Mapping** — mapping observed pipeline gaps to GDPR principles, EU AI Act obligations, and accountability requirements
4. **Governance Controls & Recommendations** — proposing concrete, actionable controls tied to specific regulatory gaps
5. **Executive Summary** — summarising key findings and required actions for NovaCred leadership

This audit builds on the findings of `01-data-quality.ipynb` (data engineering) and `02-data-analysis.ipynb` (bias detection), interpreting both through a regulatory and governance lens.

> **Regulatory Scope:** GDPR (EU) 2016/679 · EU AI Act (EU) 2024/1689

## Setup

In [2]:
import pandas as pd
import numpy as np
import hashlib
import os
import warnings
warnings.filterwarnings('ignore')

# Load the cleaned dataset produced by 01-data-quality.ipynb
df = pd.read_csv('../data/clean_credit_applications.csv')
print(f"Dataset loaded: {df.shape[0]} records, {df.shape[1]} columns")

Dataset loaded: 485 records, 33 columns


---
## Part 1 — PII Identification & Classification

### Regulatory Foundation

The GDPR (Article 4(1)) defines personal data as any information relating to an identified or identifiable natural person. This definition is intentionally broad and creates two distinct risk tiers that determine how data must be treated throughout its lifecycle.

**Direct Identifiers** are fields that identify an individual on their own, without any additional information. Their exposure immediately and unambiguously ties back to a real person.

**Quasi-Identifiers** are fields that appear innocuous in isolation but become identifying when combined. A landmark study by Sweeney (2000) demonstrated that ZIP code, gender, and date of birth in combination are sufficient to uniquely identify approximately 87% of the US population — all three of these fields are present in this dataset.

The distinction matters operationally: direct identifiers must be pseudonymised or dropped for any analytical use; quasi-identifiers require generalisation techniques such as age banding or ZIP truncation to reduce re-identification risk.

In [3]:
pii_catalog = pd.DataFrame([
    # ------------------ DIRECT IDENTIFIERS ------------------
    {
     "Field": "applicant_info.full_name",  
     "Type": "Direct Identifier",
     "Present in Clean CSV": "Yes — plaintext values",
     "GDPR Concern": "Art. 4(1): identifies individual on its own",
     "Sensitivity": "HIGH" 
    },
    {
     "Field": "applicant_info.ssn",        
     "Type": "Direct Identifier",
     "Present in Clean CSV": "Yes — plaintext values",
     "GDPR Concern": "Art. 4(1): government-issued unique ID",
     "Sensitivity": "CRITICAL"
    },
    {
     "Field": "applicant_info.email",      
     "Type": "Direct Identifier",
     "Present in Clean CSV": "Yes — plaintext values",
     "GDPR Concern": "Art. 4(1): directly identifies individual",
     "Sensitivity": "HIGH"
    },
    {
     "Field": "applicant_info.ip_address", 
     "Type": "Direct Identifier",
     "Present in Clean CSV": "Yes —  plaintext values",
     "GDPR Concern": "Art. 4(1): ECJ (Breyer) confirms IP = personal data",
     "Sensitivity": "HIGH"
    },

    # ------------------ QUASI-IDENTIFIERS ------------------
    {
     "Field": "applicant_info.date_of_birth", 
     "Type": "Quasi-Identifier",
     "Present in Clean CSV": "Yes — exact birth dates",
     "GDPR Concern": "Art. 4(1): part of re-identifying combination",
     "Sensitivity": "MEDIUM"
    },
    {
     "Field": "applicant_info.zip_code",   
     "Type": "Quasi-Identifier + Proxy",
     "Present in Clean CSV": "Yes — five-digit values",
     "GDPR Concern": "Art. 4(1): re-identifying combination",
     "Sensitivity": "MEDIUM"
    },
    {
     "Field": "applicant_info.gender",     
     "Type": "Quasi-Identifier + Protected",
     "Present in Clean CSV": "Yes",
     "GDPR Concern": "Art. 4(1) and EU AI Act (bias risk)",
     "Sensitivity": "MEDIUM"
    }
])
pii_catalog[["Field","Type", "Sensitivity", "Present in Clean CSV","GDPR Concern"]]

,Field,Type,Sensitivity,Present in Clean CSV,GDPR Concern
0,applicant_info.full_name,Direct Identifier,HIGH,Yes — plaintext values,Art. 4(1): identifies individual on its own
1,applicant_info.ssn,Direct Identifier,CRITICAL,Yes — plaintext values,Art. 4(1): government-issued unique ID
2,applicant_info.email,Direct Identifier,HIGH,Yes — plaintext values,Art. 4(1): directly identifies individual
3,applicant_info.ip_address,Direct Identifier,HIGH,Yes — plaintext values,Art. 4(1): ECJ (Breyer) confirms IP = personal...
4,applicant_info.date_of_birth,Quasi-Identifier,MEDIUM,Yes — exact birth dates,Art. 4(1): part of re-identifying combination
5,applicant_info.zip_code,Quasi-Identifier + Proxy,MEDIUM,Yes — five-digit values,Art. 4(1): re-identifying combination
6,applicant_info.gender,Quasi-Identifier + Protected,MEDIUM,Yes,Art. 4(1) and EU AI Act (bias risk)


### PII Sensitivity Classification

The sensitivity levels assigned above are derived from a risk-based assessment of two factors: the potential harm to the individual if the data is exposed (identity theft, discrimination, reputational damage) and the regulatory risk to NovaCred.

- **CRITICAL — Irrevocable Government Identifiers:** Data that uniquely identifies an individual across external systems and cannot be changed if compromised (SSN). A breach presents immediate risk of financial fraud and identity theft, requiring the strictest cryptographic protection.

- **HIGH — Direct Identifiers:** Data that identifies a specific natural person without any cross-referencing (full name, email, IP address). Exposure unambiguously ties back to a real individual and triggers full GDPR data subject rights obligations.

- **MEDIUM — Quasi-Identifiers & Protected Attributes:** Fields that appear innocuous in isolation but become highly identifying in combination — the "Mosaic Effect". As demonstrated by Sweeney (2000), ZIP code, gender, and DOB together can uniquely identify 87% of the US population. Gender additionally requires special handling as a protected attribute under EU AI Act Art. 10, which mandates examination of training data for bias against protected groups.

---
### 1.1 — Quasi-Identifier Re-identification Risk

Removing the four direct identifiers is a necessary first step — but it is not sufficient. The cell below tests empirically whether the quasi-identifiers alone can re-identify applicants, using the methodology used in Sweeney (2000).

In [4]:
quasi_cols  = ["applicant_info.zip_code", "applicant_info.gender", "applicant_info.date_of_birth"]
df_quasi    = df[quasi_cols].dropna()
total       = len(df_quasi)
unique      = df_quasi.drop_duplicates().shape[0]
re_id_rate  = unique / total * 100

print(f"Records with all three quasi-identifiers : {total}")
print(f"Unique (ZIP, Gender, DOB) combinations   : {unique}")
print(f"Re-identification rate                   : {re_id_rate:.1f}%")

Records with all three quasi-identifiers : 485
Unique (ZIP, Gender, DOB) combinations   : 485
Re-identification rate                   : 100.0%


> **Finding:** Every applicant in this dataset is uniquely identifiable using only ZIP code, gender, and date of birth — a 100% re-identification rate. Therefore, Quasi-identifier **generalisation** (age banding, ZIP truncation) is strictly required before any analytical export. Because exact street-level geography and specific birth dates are not legally necessary nor predictive requirements for a credit scoring model, this generalisation represents an acceptable loss of data utility in exchange for privacy compliance. Both mitigations are applied in Section 2.

---
## Part 2 — Pseudonymization Demonstration

### Regulatory Foundation

GDPR Article 4(5) defines pseudonymisation as processing personal data so it can no longer be attributed to a specific data subject without additional information kept separately. Three techniques exist — the choice depends on whether the original value needs to be recoverable for authorised purposes (e.g., fulfilling a GDPR Art. 17 erasure request):

| Technique | Reversible? | Still Personal Data? | Key Weakness |
|-----------|-------------|---------------------|--------------|
| **Plain SHA-256 hashing** | No | Yes | **Deterministic** — same input always yields same output. An attacker who knows the input space (e.g. all valid SSN formats) can pre-compute every possible hash and match against the dataset. No lookup table required for the attack. |
| **Salted SHA-256 hashing** | No (without salt) | Yes | Salt must be stored securely and separately. If the salt is exposed alongside the data, the protection collapses. Appropriate when recovery is not operationally needed. |
| **Tokenization** | Yes (via lookup table) | Yes | Lookup table must be secured; if stolen, all tokens are reversed. Enables GDPR Art. 17 erasure: delete the token from the table and the record becomes effectively anonymous. |

**Decision applied in this notebook:**
- **SSN, email, IP address** → Salted SHA-256. These fields have no business reason to be recovered after pseudonymisation. The salt makes pre-computation attacks infeasible.
- **Full name** → Tokenization. May need to be recovered by a compliance officer responding to a GDPR Art. 15 access or Art. 17 erasure request. A secured lookup table enables this without exposing the name in the analytical dataset.

> **Governance Audit Note: Display of Plaintext Identifiers**
> 
> For the sole purpose of this audit demonstration, the original plaintext identifiers (e.g., `applicant_info.ssn`, `applicant_info.full_name`) are temporarily retained and displayed alongside their pseudonymised outputs to verify the efficacy of the transformation. 
>
> **Production Control:** In a live production pipeline, these original plaintext columns must be immediately dropped from the analytical dataframe (`df.drop()`) following transformation. As mentioned above, the original names and their corresponding tokens would be written to a highly restricted, access-controlled lookup table (Token Vault) to ensure the analytical dataset remains privacy-compliant.

In [5]:
# ── Salted SHA-256 for SSN, email, IP ───────────────────────────────────────
# In production: SALT = os.environ.get("NOVACRED_PSEUDONYM_SALT")
# The salt is generated once per deployment and stored in a secrets manager —
# never in code, version control, or alongside the data.
SALT = os.urandom(16).hex()

def pseudonymize(value, salt):
    """
    Salted SHA-256 pseudonymisation.
    The salt breaks the deterministic link between input and output, making
    pre-computation (rainbow table) attacks infeasible even when the attacker
    knows all possible input values (e.g., all valid SSN formats).
    """
    if pd.isna(value):
        return None
    return hashlib.sha256((salt + str(value)).encode()).hexdigest()

df["ssn_pseudonymized"]   = df["applicant_info.ssn"].apply(lambda x: pseudonymize(x, SALT))
df["email_pseudonymized"] = df["applicant_info.email"].apply(lambda x: pseudonymize(x, SALT))
df["ip_pseudonymized"]    = df["applicant_info.ip_address"].apply(lambda x: pseudonymize(x, SALT))

# ── Tokenization for full name ────────────────────────────────────────────────
# Reversible via a lookup table stored separately under strict access controls.
# Supports Art. 17 erasure: deleting a token from the table makes the record
# effectively anonymous without touching the analytical dataset.
unique_names   = df["applicant_info.full_name"].dropna().unique()
name_to_token  = {name: f"APPLICANT_{str(i+1).zfill(3)}" for i, name in enumerate(unique_names)}
df["name_token"] = df["applicant_info.full_name"].map(name_to_token)

print(f"Pseudonymisation complete. Sample output (first 3 records):")
df[["applicant_info.ssn", "ssn_pseudonymized",
    "applicant_info.full_name", "name_token"]].head(3)

Pseudonymisation complete. Sample output (first 3 records):


,applicant_info.ssn,ssn_pseudonymized,applicant_info.full_name,name_token
0,596-64-4340,f6bfda824a5f9688b4d18bd6c09a0408f2e9a1090cbb7e...,Jerry Smith,APPLICANT_001
1,425-69-4784,d8e17cb7cd59a7299a4ccffa8779f4c2e403cd214f4c5d...,Brandon Walker,APPLICANT_002
2,370-78-5178,7224df199be8664eca2d6aeb4fdd88860c5eddd268f39d...,Scott Moore,APPLICANT_003


> **GDPR Art. 4(5) note:** Pseudonymised data — whether hashed or tokenized — **remains personal data**. It can be re-linked using the salt or lookup table. Data subjects retain all rights: access (Art. 15), rectification (Art. 16), erasure (Art. 17), and objection (Art. 21). Pseudonymisation reduces risk; it does not reduce compliance obligations.


### 2.1 — Constructing the Privacy-Safe Analytical Dataset

The cell below assembles the version of the dataset that should be used for all downstream modelling and analysis. It applies all privacy controls in a single reproducible transformation: dropping plaintext direct identifiers, substituting pseudonymised versions, generalising exact DOB to age bands, truncating ZIP to 3-digit prefix, and removing spending categories with no documented lawful basis.

In [6]:
# 2.1 — Constructing the Privacy-Safe Analytical Dataset

df_analytical = df.copy()

# 1. Drop plaintext direct identifiers
df_analytical.drop(columns=[
    "applicant_info.full_name", "applicant_info.ssn",
    "applicant_info.email",     "applicant_info.ip_address",
], inplace=True)

# 2. Retain pseudonymised references
df_analytical["name_token"] = df["name_token"]
df_analytical["ssn_hash"]   = df["ssn_pseudonymized"]

# 3. Generalise exact DOB to age band
ref = pd.Timestamp("2024-01-01")
dob = pd.to_datetime(df["applicant_info.date_of_birth"], errors="coerce")
df_analytical["age"] = dob.apply(
    lambda x: ref.year - x.year - ((ref.month, ref.day) < (x.month, x.day))
    if pd.notna(x) else np.nan
)
df_analytical["age_band"] = pd.cut(
    df_analytical["age"],
    bins=[17, 25, 40, 60, 100],
    labels=["18-25", "26-40", "41-60", "61+"]
)
df_analytical.drop(columns=["applicant_info.date_of_birth", "age"], inplace=True)

# 4. Truncate ZIP to 3-digit prefix
df_analytical["zip_prefix"] = df["applicant_info.zip_code"].astype(str).str[:3]
df_analytical.drop(columns=["applicant_info.zip_code"], inplace=True)

# 5. Drop sensitive spending categories
df_analytical.drop(columns=[
    "spending_Adult Entertainment", "spending_Gambling", "spending_Alcohol"
], inplace=True)

print(f"Original dataset   : {df.shape[1]} columns")
print(f"Analytical dataset : {df_analytical.shape[1]} columns")
print(f"Direct PII removed : full_name, SSN, email, IP address")
print(f"Quasi-IDs treated  : DOB → age_band | 5-digit ZIP → 3-digit prefix")
print("-" * 65)

# Display the first 5 rows to provide visual context of the safe dataset
df_analytical.head()

Original dataset   : 37 columns
Analytical dataset : 31 columns
Direct PII removed : full_name, SSN, email, IP address
Quasi-IDs treated  : DOB → age_band | 5-digit ZIP → 3-digit prefix
-----------------------------------------------------------------


,_id,processing_timestamp,applicant_info.gender,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,...,spending_Transportation,spending_Travel,spending_Utilities,ssn_pseudonymized,email_pseudonymized,ip_pseudonymized,name_token,ssn_hash,age_band,zip_prefix
0,app_200,2024-01-15 00:00:00+00:00,Male,73000.0,23,0.20,31212,False,algorithm_risk_score,NaN,...,0.0,0.0,0.0,f6bfda824a5f9688b4d18bd6c09a0408f2e9a1090cbb7e...,d58765543cf5eb06954bf680605cb28b1f5d3df9efaf81...,58b1150053a082726ac595c3b3d27d270ea24fa55ac648...,APPLICANT_001,f6bfda824a5f9688b4d18bd6c09a0408f2e9a1090cbb7e...,18-25,100
1,app_037,NaN,Male,78000.0,51,0.18,17915,False,algorithm_risk_score,NaN,...,0.0,0.0,0.0,d8e17cb7cd59a7299a4ccffa8779f4c2e403cd214f4c5d...,ff6c46f45b4d8dd70b65bc1028f73fc51d5d4986242320...,5f05c9248e59c15d7b0615c15ab0435fc8990b785126c4...,APPLICANT_002,d8e17cb7cd59a7299a4ccffa8779f4c2e403cd214f4c5d...,26-40,100
2,app_215,NaN,Male,61000.0,41,0.21,37909,True,NaN,vacation,...,0.0,0.0,0.0,7224df199be8664eca2d6aeb4fdd88860c5eddd268f39d...,9acaa088af0dec659b15d660285a2b43ccabdbb3a6610f...,211d6a3798533927694ff08fd130ed9294630ce2b2f3f5...,APPLICANT_003,7224df199be8664eca2d6aeb4fdd88860c5eddd268f39d...,26-40,100
3,app_024,NaN,Male,103000.0,70,0.35,0,True,NaN,NaN,...,0.0,0.0,0.0,c22b3581f9564363c4b3d9be57028b9df582244b479442...,a899fe4157cc3ecebff05065ff385b2af8856b01fa5854...,c6d5e290ce05ac2708820f380ffffbd6b1abd81be11fca...,APPLICANT_004,c22b3581f9564363c4b3d9be57028b9df582244b479442...,26-40,100
4,app_184,2024-01-15 00:00:00+00:00,Male,57000.0,14,0.23,31763,False,algorithm_risk_score,NaN,...,0.0,0.0,0.0,b051a74e6ef62506f0cc96b3272eadec7f63f7673ba045...,b063d5baa7026dba08d0ae34bac5d1a44fbf8fe86a71e9...,ed905eaa3b6a265c4ff3287eeb12eeb5e7284c8509bc51...,APPLICANT_005,b051a74e6ef62506f0cc96b3272eadec7f63f7673ba045...,18-25,100


### 2.2 — k-Anonymity Verification

In Section 1.1, we proved empirically that the original combination of exact ZIP code, gender, and date of birth resulted in a 100% re-identification risk. 

While we just applied our mitigations in Section 2.1 by generalising those fields into `age_band` and `zip_prefix`, data governance requires proof, not assumptions. We must mathematically verify that our transformations successfully destroyed the uniqueness of those records. We do this by measuring **k-Anonymity**.

**What is k-Anonymity?** It is a privacy model requiring that every individual in a dataset is indistinguishable from at least *k-1* other individuals, based solely on their quasi-identifiers. The transformations we just applied group people into "equivalence classes" (e.g., all Males, aged 26-40, in ZIP prefix 100).

**Setting the Governance Threshold ($k \ge 5$):**
While privacy regulations do not mandate a universal $k$ value, relying on group sizes of 2 or 3 still leaves individuals vulnerable to background knowledge attacks. Therefore, for this audit, NovaCred has established an internal governance threshold of **$k \ge 5$**. 
* Any group where $k < 5$ is deemed to have an unacceptably small cohort, meaning anonymity cannot be guaranteed. 
* These specific at-risk rows must be addressed before the dataset is cleared for downstream modelling.

**The Governance Check:** The code below audits our newly transformed analytical dataset to verify the minimum *k* value achieved. NovaCred's internal compliance target dictates that **k ≥ 5** for all groups before this dataset can be cleared for downstream modelling.

In [7]:
quasi_id_cols = ["zip_prefix", "applicant_info.gender", "age_band"]

k_groups = (df_analytical
    .groupby(quasi_id_cols, observed=True)
    .size()
    .reset_index(name="group_size")
    .sort_values("group_size"))

min_k    = k_groups["group_size"].min()
max_k    = k_groups["group_size"].max()
mean_k   = k_groups["group_size"].mean()
below_5  = (k_groups["group_size"] < 5).sum()
total_g  = len(k_groups)

print(f"k-Anonymity Results")
print(f"{'='*50}")
print(f"Minimum k (least anonymous group) : {min_k}")
print(f"Maximum k (most anonymous group)  : {max_k}")
print(f"Average k across all groups       : {mean_k:.1f}")
print(f"Groups with k < 5 (at-risk)       : {below_5} of {total_g}")
print()
if below_5 > 0:
    print("At-risk groups (k < 5):")
    print(k_groups[k_groups["group_size"] < 5].to_string(index=False))

k-Anonymity Results
Minimum k (least anonymous group) : 1
Maximum k (most anonymous group)  : 118
Average k across all groups       : 23.1
Groups with k < 5 (at-risk)       : 9 of 21

At-risk groups (k < 5):
zip_prefix applicant_info.gender age_band  group_size
       100                Female    18-25           1
       300                  Male      61+           1
       300                Female      61+           1
       300                  Male    18-25           2
       300                Female    18-25           2
       300                Female    41-60           2
       300                  Male    26-40           3
       300                Female    26-40           3
       300                  Male    41-60           4


> **Finding & Required Mitigation:** Generalising exact DOB to age bands and truncating ZIP codes successfully raised the average $k$ to 23.1, a massive improvement over the raw dataset (where $k=1$ for every record). However, 9 specific groups still fall below the $k \ge 5$ threshold, representing small demographic cohorts that are insufficiently anonymous for external sharing. 
> 
> Further downstream processing must be protected using **Differential Privacy**. This framework provides a mathematical guarantee that an individual's presence or absence in the dataset does not significantly change the output. We recommend injecting calibrated, reasonable noise into sensitive attributes using the **Laplace Mechanism** for continuous data or the **Coin Flip Trick** for categorical data. 
>
> **Implementation Deferral:** The mathematical execution of Differential Privacy is intentionally deferred from this baseline governance audit. Tuning the **Epsilon ($\epsilon$) Parameter** cannot be done in isolation; it requires evaluating the strict trade-off between privacy and predictive utility. As the Governance Officer, the mandate is established here, but the exact calibration of noise must be collaboratively determined and applied by the Data Science team during the model engineering phase.

---
## Part 3 — Regulatory Compliance Mapping

Compliance is not a single checklist but a layered obligation spanning two distinct regulatory frameworks. This section maps the gaps identified across the NovaCred pipeline to their specific legal consequences under each:

- **Section 3.1 — GDPR:** Maps data quality and pipeline failures to violations of the seven data protection principles in Art. 5(1). All findings reference the **raw ingestion baseline** (502 records), which represents the state of the pipeline before any engineering remediation — the legally relevant moment at which violations occur.
- **Section 3.2 — EU AI Act:** Classifies NovaCred's credit scoring system under the Act's risk-based framework and audits compliance against the mandatory obligations that classification triggers.
- **Section 3.3 — Accountability Check:** A programmatic audit verifying that the governance metadata fields required to *prove* compliance under GDPR Art. 5(2) and EU AI Act Art. 13 are actually present in the dataset.

### 3.1 — Mapping Data Quality Dimensions to GDPR Principles

GDPR grants individuals powerful rights over their data — including the Right to Rectification (Art. 16) and the Right to Erasure (Art. 17). Addressing data quality failures is a mandatory prerequisite for these rights: we cannot correct or delete a specific individual's data if the pipeline cannot reliably identify which records belong to them.

The findings below are sourced from `01-data-quality.ipynb` and interpreted here through a regulatory lens. All figures refer to the **raw ingestion baseline** (502 records).

---

**1. Exposure of Direct Identifiers — Art. 5(1)(c) & Art. 5(1)(f)**
- **The Finding:** The raw pipeline exports plaintext SSNs, full names, emails, and IP addresses directly into the analytical dataset with no transformation.
- **The GDPR Violation:** Violates **Art. 5(1)(f) Integrity & Confidentiality** — failure to secure personal data against unauthorised access — and **Art. 5(1)(c) Data Minimisation** — processing more data than is necessary for the stated purpose. Direct identifiers have zero predictive value for creditworthiness assessment and have no analytical justification for inclusion.

---

**2. Missing Processing Timestamps — Art. 5(1)(e)**
- **The Finding:** 440 of 502 raw records (88%) are missing a `processing_timestamp`.
- **The GDPR Violation:** Violates **Art. 5(1)(e) Storage Limitation**. Without knowing when a record entered the system, it is mathematically impossible to calculate legal retention periods or enforce any deletion schedule. NovaCred cannot demonstrate it is not holding personal data longer than permitted.

---

**3. Processing Without Lawful Basis — Art. 6, Art. 5(1)(b) & Art. 5(1)(c)**
- **The Finding:** The pipeline collects sensitive lifestyle categories (Gambling, Adult Entertainment) with no documented relationship to credit risk. Furthermore, the raw ingestion baseline contains no metadata or fields (e.g., consent flags) indicating a legal right to process any of the 502 records.
- **The GDPR Violation:** This is a fundamental breach of Art. 6 (Lawfulness of Processing). Processing is prohibited unless a specific lawful basis is established. Additionally, collecting lifestyle data without a specific justification violates Art. 5(1)(b) Purpose Limitation and Art. 5(1)(c) Data Minimisation. No such justification exists here.

---

**4. Accuracy & Uniqueness Failures — Art. 5(1)(d)**
- **The Finding:** 157 inconsistent date formats; 3 SSN values are shared across 6 distinct application records; 4 malformed email addresses; 2 future-dated processing timestamps; 2 records with negative credit history months; 2 records with underage applicants; 1 record with a negative savings balance.
- **The GDPR Violation:** Violates **Art. 5(1)(d) Accuracy**, which requires data to be correct and kept up to date. Inconsistent date formats mean NovaCred cannot reliably verify applicant age or calculate legal retention periods based on birth dates. Shared SSNs indicate either fraud or a pipeline failure — NovaCred cannot guarantee which data belongs to which real individual. Logically impossible values mean any model trained on this data ingests corrupt inputs, which additionally violates the EU AI Act Art. 10 requirement that training data be free of errors to the best extent possible.

### 3.2 — EU AI Act: High-Risk Classification & Obligations

The EU AI Act (Regulation 2024/1689) establishes a risk-based classification framework for AI systems. **Credit scoring and creditworthiness assessment is explicitly enumerated as a High-Risk AI system in Annex III, Point 5(b).** This classification is not discretionary — it applies to NovaCred's automated loan approval system by virtue of what it does.

High-Risk classification triggers six mandatory obligations. Three can be directly assessed from
the dataset and pipeline outputs — and all three show non-compliance. The remaining three require
access to system documentation, model code, or pre-deployment records that fall outside the scope
of a dataset-level audit; they are noted here as obligations NovaCred must satisfy independently
before any deployment.

| Article | Obligation | Compliance Status |
| :--- | :--- | :--- |
| **Art. 9** | Establish and maintain a risk management system throughout the AI lifecycle | **Cannot audit from dataset** |
| **Art. 10** | Training data examined and corrected for biases that could lead to discrimination | **NON-COMPLIANT** |
| **Art. 13** | Each decision must be traceable to the specific model version that produced it | **NON-COMPLIANT** |
| **Art. 14** | Natural persons able to oversee, override, or reverse automated outputs | **NON-COMPLIANT** |
| **Art. 15** | System achieves appropriate levels of accuracy, robustness, and cybersecurity | **Cannot audit from dataset** |
| **Art. 43** | Conformity assessment completed and filed before system is put into service | **Cannot audit from dataset** |



> **On the three unauditable articles:** The absence of evidence is not evidence of compliance.
> Art. 9 requires documented risk management procedures maintained throughout the system
> lifecycle — this must be demonstrated through internal governance documentation. Art. 15
> requires model-level validation of accuracy and robustness against adversarial inputs — this
> requires access to model performance metrics and test results. Art. 43 requires a conformity
> assessment to be completed and filed before the system is put into service — whether this has
> occurred cannot be determined from the dataset alone and must be verified through NovaCred's
> pre-deployment records.

---

**Article 10 — Training Data Quality & Bias Mitigation**

Art. 10 requires that training data be relevant, representative, and free of errors to the best extent possible. It explicitly mandates the detection and mitigation of biases that could lead to risks to fundamental rights. Our analysis in `02-data-analysis.ipynb` identifies three systemic violations:

- **Gender Disparate Impact:** Female applicants are approved at **51.2%** vs. **66.3%** for male applicants — a Disparate Impact ratio of **0.77**, below the 0.80 four-fifths threshold at which adverse impact is presumed. The difference is statistically significant (proportions z-test, `02-data-analysis.ipynb`).

- **Age-Based Disparity:** Using the age bands defined in `02-data-analysis.ipynb`, Gen Z applicants (18–25) are approved at only **32.5%**, compared to 57.7% for Millennials — a **25.2 percentage point gap**, confirmed significant by Chi-Square test (p = 0.002). Applying a single model uniformly across age groups with structurally different credit histories constitutes Aggregation Bias.

- **Proxy Discrimination:** ZIP code acts as a near-perfect proxy for gender in this dataset — NYC 100xx area is 89% Male with a 64.6% approval rate; LA 902xx area is 93% Female with a 52.4% approval rate, as validated in `02-data-analysis.ipynb`. Additionally, `credit_history_months` , `annual_income` and `savings_balance` are found to be strongly correlated with age: younger applicants are structurally disadvantaged by this feature independent of actual creditworthiness, constituting Measurement Bias.

---

**Article 13 — Transparency & Traceability**

Art. 13 requires that High-Risk AI systems be sufficiently transparent to allow the deployer to interpret their outputs. In practice this means every automated decision must be traceable to the specific model version that produced it — enabling audit, rollback, and accountability. There is no field that enables the required traceability and transparency in the dataset.

---

**Article 14 — Human Oversight**

Art. 14 requires that systems be designed to allow natural persons to oversee the AI, including the technical ability to ignore, override, or reverse an automated decision. The most common rejection reason in the dataset is `algorithm_risk_score`, confirming fully automated decisions are already in operation. The `human_review_flag` field is entirely absent, meaning no mechanism exists to route contested or borderline decisions to a human underwriter.


### 3.3 — Mandatory Governance Fields: Accountability Check
GDPR Art. 5(2) — the Accountability Principle — places the burden of proof entirely on the data controller. NovaCred must not merely claim compliance; it must be able to **demonstrate** it at any time through documentation and audit trails to answer fundamental governance questions: 

*When did the applicant consent? Why do we have this data? When must we delete it?* 

The following programmatic audit checks the baseline dataset for the presence of six critical governance fields.

In [8]:
governance_fields = {
    "consent_timestamp"  : "Lawfulness & Transparency — GDPR Art. 5(1)(a) & Art. 6",
    "processing_purpose" : "Purpose Limitation — GDPR Art. 5(1)(b)",
    "retention_until"    : "Storage Limitation — GDPR Art. 5(1)(e)",
    "data_source"        : "Transparency — GDPR Art. 14",
    "human_review_flag"  : "Automated Decision-Making — GDPR Art. 22",
    "model_version"      : "AI System Traceability — EU AI Act Art. 13"
}

results = pd.DataFrame([
    {
        "Field": field,
        "Present in Dataset": "Yes" if field in df.columns else "MISSING",
        "Regulatory Basis": basis
    }
    for field, basis in governance_fields.items()
])

missing = (results["Present in Dataset"] == "MISSING").sum()
results

,Field,Present in Dataset,Regulatory Basis
0,consent_timestamp,MISSING,Lawfulness & Transparency — GDPR Art. 5(1)(a) ...
1,processing_purpose,MISSING,Purpose Limitation — GDPR Art. 5(1)(b)
2,retention_until,MISSING,Storage Limitation — GDPR Art. 5(1)(e)
3,data_source,MISSING,Transparency — GDPR Art. 14
4,human_review_flag,MISSING,Automated Decision-Making — GDPR Art. 22
5,model_version,MISSING,AI System Traceability — EU AI Act Art. 13


> **Finding:** All six mandatory governance fields are absent from the dataset. This is not a minor data quality issue — it is a systemic architectural failure. Without these fields, NovaCred cannot respond to a GDPR Subject Access Request (Art. 15), enforce its own retention policy (Art. 5(1)(e)), demonstrate a lawful basis for processing (Art. 6), or prove which model version produced any given credit decision (EU AI Act Art. 13). The pipeline is currently operating in a state where NovaCred cannot currently demonstrate lawful processing under GDPR Art. 5(2).

---
## Part 4 — Governance Controls & Recommendations

The four controls below directly address the gaps identified in Part 3. Each is mapped to the specific regulatory obligation it satisfies and ordered by implementation urgency.

---

### Control 1 — Privacy by Design Pipeline Restructuring
**Regulatory basis:** GDPR Art. 25 (Privacy by Design), Art. 5(1)(c) (Data Minimisation)

**Priority:** Immediate — before any model training or retraining

**Current state:** Raw PII flows from JSON ingestion through to the analytical CSV without any transformation. Plaintext SSNs, names, emails, and IP addresses are accessible to anyone with repository access.

**Required change:** Pseudonymisation must occur at ingestion, before any data is written to storage. The pipeline should produce two distinct, separately managed outputs:

| Output | Contents | Access |
| :--- | :--- | :--- |
| **Secure PII store** | Token → {full_name, SSN, email, IP, exact DOB} | Compliance team only; encrypted at rest; supports Art. 15 access and Art. 17 erasure |
| **Analytical dataset** | Token, age_band, zip_prefix, financial fields, decision fields | Data science and governance teams; version-controlled; safe for modelling |

---

### Control 2 — Mandatory Record-Level Governance Fields
**Regulatory basis:** GDPR Arts. 5(1)(b), 5(1)(e), 6(1), Art. 14; EU AI Act Art. 13

**Priority:** Implement before next data collection cycle

The six fields identified in Section 3.3 must be added to the data schema with ingestion-level
validation. Records missing any mandatory field must be quarantined before entering the decision
pipeline — their absence means NovaCred cannot demonstrate lawful processing under GDPR Art. 5(2)
at any point in the data lifecycle.

**`consent_timestamp` — datetime**
GDPR Art. 6(1) prohibits all processing unless at least one lawful basis is established before
data is collected. For a consumer credit application, the most natural basis is Art. 6(1)(a)
explicit consent, supported by Art. 7 which requires the controller to demonstrate that consent
was freely given, specific, and informed. Without a timestamp recording exactly when and how
consent was obtained, NovaCred cannot respond to a regulator demanding proof of lawful collection,
nor can it respond to a data subject exercising their Art. 7(3) right to withdraw consent. The
field is not optional — it is the documentary foundation for the entire processing chain.

**`processing_purpose` — string (enum: `credit_assessment` | `fraud_detection` |
`regulatory_reporting`)**
GDPR Art. 5(1)(b) requires that data be collected for a specified, explicit, and legitimate
purpose and not further processed in a way incompatible with that purpose. Without a
`processing_purpose` field, there is no machine-readable record of why any given record was
collected, making it impossible to enforce purpose limitation programmatically or to answer an
Art. 15 Subject Access Request accurately. It also prevents NovaCred from demonstrating that
secondary uses — such as model retraining or fraud analysis — are compatible with the original
collection purpose, as required by Art. 6(4).

**`retention_until` — date**
GDPR Art. 5(1)(e) requires that personal data be kept no longer than necessary for the purpose
for which it was collected. However, GDPR does not itself specify universal retention periods —
these must be defined by NovaCred's legal counsel in accordance with applicable sectoral law
(for example, the EU Anti-Money Laundering Directive requires a minimum five-year retention
period after a business relationship ends). The `retention_until` field is the mechanism by
which that legal determination is enforced in code: it is calculated at ingestion once the
retention policy is defined, and a scheduled deletion job uses it to identify records that must
be deleted or anonymised. Without this field, 88% of records in the current dataset have no
processing timestamp at all, making it mathematically impossible to calculate or enforce any
retention schedule.

**`data_source` — string (enum: `web_portal` | `mobile_app` | `branch` | `third_party`)**
GDPR Art. 14 requires that when personal data is not collected directly from the data subject —
for example when sourced from a broker, credit reference agency, or partner platform — the
controller must inform the individual of this fact, including what data was obtained and the
source. Without a `data_source` field, NovaCred cannot identify which records arrived via third
parties and therefore cannot determine which applicants are entitled to an Art. 14 notification.
It also prevents any forensic investigation into which upstream sources introduced the data
quality failures identified in `01-data-quality.ipynb`.

**`human_review_flag` — boolean (default: False)**
GDPR Art. 22(1) grants data subjects the right not to be subject to decisions based solely on
automated processing that produce legal or similarly significant effects — and denial of credit
qualifies. Art. 22(2) permits automated decisions only if the data subject has given explicit
consent, or if the decision is necessary for a contract, but in both cases Art. 22(3) requires
the controller to implement measures to safeguard the data subject's rights, including the right
to obtain human intervention. The current dataset shows that 163 of 202 rejections (80.7%) carry
the reason `algorithm_risk_score` — fully automated, with no documented human involvement.
The `human_review_flag` field creates the audit trail proving which decisions involved human
oversight and which did not, enabling the three-tier oversight model proposed in Control 4.

**`model_version` — string (e.g. `novacred-v1.2.3`)**
EU AI Act Art. 13 requires that High-Risk AI systems be sufficiently transparent to allow the
deployer to interpret and use their outputs. In practice, this requires that every automated
decision be traceable to the specific model version that produced it — enabling rollback if a
model version is found to be biased or erroneous, supporting audit and conformity assessment
under Art. 43, and allowing regulators to reconstruct which algorithm was in operation during
any period under investigation. Without this field, it is impossible to determine which model
produced any given credit decision in the dataset, making post-deployment accountability
entirely unenforceable.

---

> **Note on retention periods:** Specific retention durations for each PII field are not
> prescribed in this audit. GDPR Art. 5(1)(e) requires data be held no longer than necessary
> for the stated purpose, but the exact periods must be defined by NovaCred's legal counsel
> in accordance with applicable sectoral law, then enforced programmatically via the
> `retention_until` field above.

---

### Control 3 — Bias Monitoring & Feature Governance
**Regulatory basis:** EU AI Act Art. 10 (Data Governance), Art. 9 (Risk Management)

**Priority:** Before any retraining or new model deployment

- **Pre-Deployment Bias Gate:** Compute the Disparate Impact ratio on a holdout set before every deployment. Block deployment if DI < 0.80 for any protected attribute unless a documented business necessity justification is filed with the compliance team.
- **ZIP Code & Credit History Exclusion:** Both features must be removed from the model feature set. ZIP code is a near-perfect proxy for gender; credit history length is a structural proxy for age. Their inclusion reproduces discriminatory outcomes even when protected attributes are explicitly excluded.
- **Model Card per Deployment:** Each model version must document training data period, DI ratios at validation, full feature list, and any features excluded for fairness reasons — satisfying EU AI Act Art. 13 transparency requirements.

---

### Control 4 — Human Oversight Implementation
**Regulatory basis:** GDPR Art. 22, EU AI Act Art. 14
**Priority:** Before any further production deployment

NovaCred must transition from a binary "Black Box" auto-rejection model to a three-tier governance architecture:

1.  **Tier 1: Automated Approval (Green-Light):** For applications that clearly exceed all credit and fairness thresholds.
2.  **Tier 2: Assisted Review (Yellow-Light):** Mandatory human intervention triggered if an applicant is flagged by a **proxy variable** (like ZIP code) or belongs to a group with a **Disparate Impact ratio < 0.8**. This ensures a human underwriter can override biased algorithmic suggestions.
3.  **Tier 3: Full Manual Review (Red-Light):** Applicants must have a clear pathway to contest an automated rejection, as required by **GDPR Art. 22**.

Integrate the `human_review_flag` field mentioned in Control 2 to record the tier each application passed through. This creates the audit trail required by EU AI Act Art. 9 and satisfies the applicant's right to human intervention under GDPR Art. 22.

**Audit Conclusion:** The NovaCred credit scoring model cannot proceed to CE marking or deployment until the bias identified in `02-data-analysis.ipynb` is mathematically mitigated and the `human_review_flag` is integrated into the production pipeline.

---
## Executive Summary

This audit reviewed the NovaCred credit application pipeline across two regulatory frameworks: the GDPR and the EU AI Act. The system is classified as **High-Risk AI** under EU AI Act Annex III, Point 5(b) and is currently **non-compliant with both regulations**.

### Key Findings

| # | Finding | Regulatory Impact |
| :--- | :--- | :--- |
| 1 | All 4 direct identifiers (name, SSN, email, IP) in plaintext | GDPR Art. 5(1)(f) — CRITICAL |
| 2 | Sensitive data collected without documented lawful basis | GDPR Art. 5(1)(b)(c) — HIGH |
| 3 | 88% of records missing `processing_timestamp`; retention periods unenforceable | GDPR Art. 5(1)(e) — HIGH |
| 4 | Shared SSNs, malformed emails, impossible values in raw pipeline | GDPR Art. 5(1)(d) — MEDIUM |
| 5 | Gender DI ratio = 0.77; Gen Z approval rate = 32.5% vs 57.7% for Millennials | EU AI Act Art. 10 — CRITICAL |
| 6 | ZIP code and credit history are structural proxies for protected attributes | EU AI Act Art. 10 — HIGH |
| 7 | No human oversight mechanism; fully automated rejections already in operation | GDPR Art. 22 / EU AI Act Art. 14 — HIGH |
| 8 | All 6 mandatory governance fields absent; lawful processing cannot be demonstrated | GDPR Art. 5(2) / EU AI Act Art. 13 — CRITICAL |

### Immediate Actions Required

| Priority | Action | Regulatory Driver |
| :--- | :--- | :--- |
| 1 | **Implement pseudonymisation at ingestion.** The pipeline applies no transformation to PII before writing to storage or version control, meaning every downstream file inherits the raw identifiers. | No pseudonymised column exists anywhere in the clean dataset; all identifiers are verbatim from source | GDPR Art. 25 — Privacy by Design |
| 2 | **Establish and document a lawful basis for processing before any reprocessing or retraining.** No consent timestamp, contract reference, or processing purpose exists for any of the 502 records. All downstream processing rests on an undocumented legal foundation. | `consent_timestamp` and `processing_purpose` both absent from all 485 records | GDPR Art. 6(1) — Lawfulness of Processing |
| 3 | **Exclude ZIP code from all model features.** ZIP code correlates with gender at r = 0.783, producing near-identical approval disparities to using gender directly. Its presence in any model reproduces the gender bias covertly. | NYC (100xx): 89% male, 64.6% approval vs. LA (902xx): 93% female, 52.4% approval | EU AI Act Art. 10 — Bias Mitigation |
| 4 | **Exclude credit history months from all model features pending age-proxy investigation.** Credit history correlates with age at r = 0.659, systematically disadvantaging younger applicants independent of actual credit risk. | Gen Z (18–25) approval rate: 32.5% vs. 57.7% for Millennials; credit history ~ age r = 0.659 | EU AI Act Art. 10 — Bias Mitigation |
| 5 | **Add six mandatory governance fields with ingestion-level validation.** All six fields required to demonstrate lawful processing under GDPR Art. 5(2) are entirely absent. Records missing any mandatory field should be quarantined before entering the decision pipeline. | `consent_timestamp`, `processing_purpose`, `retention_until`, `data_source`, `human_review_flag`, `model_version` all absent | GDPR Art. 5(2) — Accountability |
| 6 | **Implement human review before any further automated rejections.** 163 of 202 rejections (80.7%) carry the reason `algorithm_risk_score` — fully automated decisions with no documented human oversight and no mechanism to contest or appeal. | `human_review_flag` absent; `algorithm_risk_score` is the rejection reason for 80.7% of all rejections | GDPR Art. 22 / EU AI Act Art. 14 |
| 7 | **Cease collection of lifestyle spending categories without documented lawful basis.** Adult Entertainment, Gambling, and Alcohol spending data is currently collected with no `processing_purpose` field establishing relevance to creditworthiness. These fields should be excluded from future data collection and from any model feature set. | No `processing_purpose` field exists; these categories have no documented credit relevance | GDPR Art. 5(1)(b)(c) — Purpose Limitation & Data Minimisation |

### Regulatory Exposure Without Remediation

| Regulation | Provision | Maximum Exposure |
| :--- | :--- | :--- |
| GDPR | Art. 83(5) — critical violations of Arts. 5, 25 | €20M or 4% of global annual turnover |
| EU AI Act | Art. 99(4) — High-Risk non-compliance | €15M or 3% of global annual turnover |